# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - this time from a merged cell CSV aready created
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Set your paths here

In [ ]:
#Imports
import os
from pathlib import Path
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from scipy import stats

from plate_information import *
from plate_preprocessing import *
from mitolyso_plot_functions import *

## Debugging functions


In [ ]:
def add_drug_to_group(init_df, group, drug):
    '''
        Add the name of a drug treatment from the "Drug" column to the main "group" column
        
        Returns
            Series object: A series containing the column with the drug added to the group
    '''
    if drug is not None:
        # Replace values in 'col1' with values from 'col2' only if 'col2' is not None or NaN
        df = init_df.copy()
        df[group] = np.where(df[drug].notna(), df[drug], df[group])
        newcol = df[group]

    return newcol

#display(cell_df[["Cell_Mean_Nuclei_AreaShape_Area", "Cell_AreaShape_Area"]])



def multinucleate_cells(df):
    multinuc_df = df[df["Cell_Classify_multinucleate"] ==1]
    return multinuc_df

# cell_df_2 = enforce_objects_one_to_one(cell_df)
#cell_df_2.head()
# filter_df = enforce_objects_one_to_one(cell_df)
#print(cell_df.shape, " ", filter_df.shape)
#display(filter_df["Cell_Classify_Normal"])
def well_namer(row, col):
    '''
        Convert row and column numbers to a well name in the format A01, B02, etc.
        
        Args:
            row (int): The row number (1-8)
            col (int): The column number (1-12)
        
        Returns:
            str: Well name in the format A01, B02, etc.
    '''
    well_name = str(chr(ord('@')+ row)) + str(col).rjust(2, '0')  #make the number have a left align, adding a zero
    return well_name

def add_well_metadata(image_df):
    '''
        Add well metadata to the image DataFrame.
        
        Args:
            image_df (DataFrame): DataFrame containing image metadata
        
        Returns:
            DataFrame: Updated DataFrame with well metadata
    '''
    image_df.columns = image_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
    image_df[["Metadata_WellRow","Metadata_WellColumn","Metadata_Field"]] = image_df["Image_URL_DAPI"].str.extract(r'r(\d{2})c(\d{2})f(\d{2}).tif')
    # Convert extracted columns to int
    image_df["Metadata_WellRow"] = image_df["Metadata_WellRow"].astype(int)
    image_df["Metadata_WellColumn"] = image_df["Metadata_WellColumn"].astype(int)
    image_df["Metadata_Field"] = image_df["Metadata_Field"].astype(int)
    # apply well namer function
    image_df["Metadata_Well"] = image_df.apply(lambda x: well_namer(x["Metadata_WellRow"], x["Metadata_WellColumn"]), axis=1)
    display(image_df[["ImageNumber","Image_URL_DAPI","Metadata_WellRow","Metadata_WellColumn","Metadata_Field","Metadata_Well"]])
    
    return image_df

def update_database_with_well_metadata(db_path):
    '''
        Update the database with well metadata.
        
        Args:
            db_path (str): Path to the database file
    '''
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Read Per_Image table
    image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
    
    # Add well metadata
    updated_image_df = add_well_metadata(image_df)
    
    # Write updated DataFrame back to the database
    try:
        updated_image_df.to_sql('Per_Image', conn, if_exists='replace', index=False)
        print("Database updated successfully with well metadata.")
    except Exception as e:
        print(f"Error updating database: {e}")
    #cursor.execute("SELECT Metadata_Well FROM Per_Image LIMIT 5;")
    #cursor.fetchall()
    conn.close()

## Import the big csv

In [ ]:
#import from a giant csv
csvpath = '/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs/'
filename = 'total_combined_cell.csv'
combined_cell_df_mitolyso = pd.read_csv(os.path.join(csvpath, filename))
# filter_df = enforce_objects_one_to_one(combined_cell_df_mitolyso)
display(combined_cell_df_mitolyso.shape)
#print(cell_df.shape, " ", filter_df.shape)

#combined_cell_df_mitolyso_merged = plate_df_setup(curr_plates, curr_plate_datafolders, parent_dir, ['Cell.csv', 'Nuclei.csv','MergedMitoPerCell.csv','MergedLysoPerCell.csv'])

#display(df)


### NOTE: this is not updated to the latest shceme, will not run correctly

```python
extra_csv_path = os.path.join(csvpath,"data_with_extraRow1.csv") 

extra_data = pd.read_csv(extra_csv_path)
# Get ImageNumbers from Replicate_Number == 5 subset
rep5_image_numbers = combined_cell_df_mitolyso.loc[combined_cell_df_mitolyso["Replicate_Number"] == 5, "ImageNumber"]

# Filter extra_data to exclude those ImageNumbers
extra_data_mask = extra_data[~extra_data["ImageNumber"].isin(rep5_image_numbers)]
display(extra_data_mask.shape)
# Concatenate the filtered extra_data to the main dataframe
combined_cell_df_mitolyso = pd.concat([combined_cell_df_mitolyso, extra_data_mask], ignore_index=True)

display(combined_cell_df_mitolyso.shape)
display(combined_cell_df_mitolyso.head())```

In [ ]:
unique_combinations = combined_cell_df_mitolyso[['Replicate_Number', 'PassageNumber']].drop_duplicates()

combos = list(unique_combinations.itertuples(index=False, name=None))
display(sorted(combos))


### Filter out the poorly segmented cells and rename the columns so I don't have to change the old code

In [ ]:
display(combined_cell_df_mitolyso.shape)
filtered_df = enforce_objects_one_to_one(combined_cell_df_mitolyso)
display(filtered_df.shape)
filtered_df.columns = filtered_df.columns.str.replace(r'^Cell_', '', regex=True)
#also rename nuclei cols in the filtered df when you made the df one-to-one
filtered_df.columns = filtered_df.columns.str.replace(r"^Mean_Nuclei_", "Nuclei_", regex=True)

# Display columns containing 'Distance'
distance_cols = [col for col in filtered_df.columns if 'Area' in col]
display(distance_cols)

In [ ]:
#display(combined_cell_df_mitolyso[['Cell_AreaShape_Area', 'Cell_Children_Mitochondria_Count','Cell_Mean_Mitochondria_AreaShape_Area']])
display(filtered_df[['AreaShape_Area', 'Children_Mitochondria_Count','Mean_Mitochondria_AreaShape_Area']])


# Define the cell features


In [ ]:
# Add extra columns
def proportion_area_occupied_per_cell(df, compartment):
    # proportion of area occupied = children * mean organelle area / cell area
    colname = "Total_Area_Proportion_" + compartment + "_Per_Cell"

    # children = 'Children_' + compartment + '_Count'
    # mean_organelle_area = 'Mean_'+ compartment + '_AreaShape_Area'
    organelle_area = compartment + "_AreaShape_Area"
    cell_area = "AreaShape_Area"
    # df[colname] = df.apply(lambda x: (x[children] * x[mean_organelle_area]) / x[cell_area], axis=1)
    df[colname] = df.apply(lambda x: (x[organelle_area]) / x[cell_area], axis=1)
    return df[colname]


def mean_intensity_per_compartment_per_cell(df, compartment, name, tag, math=None):
    # Calculate the mean intensity of each compartment per cell
    # mean_intesity_per_compartment = integrated / (children*mean_area)
    colname = f"Mean_Intensity_Per_{compartment} Per_Cell"
    integrated = "Intensity_IntegratedIntensity_" + tag
    # children = 'Children_' + compartment + '_Count'
    # mean_area = 'Mean_'+ compartment + '_AreaShape_Area'
    total_organelle_area = name + "_AreaShape_Area"
    total_organelle_area = math if math is not None else total_organelle_area

    df[colname] = df.apply(lambda x: x[integrated] / x[total_organelle_area], axis=1)

    return df[colname]

# df["Mean_Mitochondria_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedMitoPerCell")
# df["Mean_Lysosomes_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(df, "MergedLysoPerCell")

# df["MeanIntensity_Lysosomes_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Lysosomes", "MergedLysoPerCell", "LAMP1")
# df["MeanIntensity_Mitochondria_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(df, "Mitochondria", "MergedMitoPerCell", "MitoTracker")

# Calculate the total area occupied by mitochondria and lysosomes per cell
def calculate_corrected_features(full_df):
    """_summary_

    Args:
        full_df (DataFrame): _description_

    Returns:
        df (DataFrame): the df with all the feature calcs
    """    
    df = full_df.copy()
    df["Math_Total_Mitochondria_AreaShape_Area_PerCell"] = (
        df["Children_Mitochondria_Count"] * df["Mean_Mitochondria_AreaShape_Area"]
    )
    df["Math_Total_Lysosomes_AreaShape_Area_PerCell"] = (
        df["Children_Lysosomes_Count"] * df["Mean_Lysosomes_AreaShape_Area"]
    )


    # Total intensity per cell based on ingegrated instensity if I don't already have the merged area
    df["Mean_Intensity_Per_Lysosomes_PerCell_Area"] = (
        mean_intensity_per_compartment_per_cell(
            df,
            "Lysosomes",
            "MergedLysoPerCell",
            "LAMP1",
            math="Math_Total_Lysosomes_AreaShape_Area_PerCell",

        )
    )
    df["Mean_Intensity_Per_Mitochondria_PerCell_Area"] = (
        mean_intensity_per_compartment_per_cell(
            df,
            "Mitochondria",
            "MergedMitoPerCell",
            "MitoTracker",
            math="Math_Total_Mitochondria_AreaShape_Area_PerCell",
        )
    )
    #same thing but for medians
    df["Median_Intensity_Per_Lysosomes_PerCell_Area"] = (
        df["Children_Lysosomes_Count"]
        * df["Mean_Lysosomes_Intensity_MeanIntensity_LAMP1"]
    )
    df["Median_Intensity_Per_Mitochondria_PerCell_Area"] = (
        df["Children_Mitochondria_Count"]
        * df["Mean_Mitochondria_Intensity_MeanIntensity_MitoTracker"]
    )


    # Corrected mitochondria and lysosomes counts per cell area (density)
    df["Density_Children_Mitochondria_Count_PerCell_Area"] = (
        df["Children_Mitochondria_Count"] / df["AreaShape_Area"]
    )
    df["Density_Children_Lysosomes_Count_PerCell_Area"] = (
        df["Children_Lysosomes_Count"] / df["AreaShape_Area"]
    )

    # Corrected organelle area fractions per cell ratio
    df["AreaFraction_Mitochondria_PerCell"] = (
        df["Math_Total_Mitochondria_AreaShape_Area_PerCell"] / df["AreaShape_Area"]
    )
    df["AreaFraction_Lysosomes_PerCell"] = (
        df["Math_Total_Lysosomes_AreaShape_Area_PerCell"] / df["AreaShape_Area"]
    )

    # mean and median area per organelle per cell
    df["Mean_Mitochondria_Area_PerCell_Area"] = (
        df["Mean_Mitochondria_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Mean_Lysosomes_Area_PerCell_Area"] = (
        df["Mean_Lysosomes_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Median_Mitochondria_Area_PerCell_Area"] = (
        df["Median_Mitochondria_AreaShape_Area"] / df["AreaShape_Area"]
    )
    df["Median_Lysosomes_Area_PerCell_Area"] = (
        df["Median_Lysosomes_AreaShape_Area"] / df["AreaShape_Area"]
    )

    # mitolyso related
    df["Children_Lysosomes_Mitochondria_Ratio"] = (
        df["Children_Lysosomes_Count"] / df["Children_Mitochondria_Count"]
    )
    df["Density_Lysosomes_Mitochondria_Ratio"] = (
        df["AreaFraction_Lysosomes_PerCell"] / df["AreaFraction_Mitochondria_PerCell"]
    )


    # Ratio of centroid distance to minimum distance for mitochondria and lysosomes
    df["Mean_Mitochondria_Distance_Centroid_Minimum_Ratio"] = (
        df["Mean_Mitochondria_Distance_Centroid_Nuclei"]
        / df["Mean_Mitochondria_Distance_Minimum_Cell"]
    )
    df["Mean_Lysosomes_Distance_Centroid_Minimum_Ratio"] = (
        df["Mean_Lysosomes_Distance_Centroid_Nuclei"]
        / df["Mean_Lysosomes_Distance_Minimum_Cell"]
    )

    # transform the mitoends - number of ends times the mean to get total per cell
    df["MitoEnds_Math_Total_NumberBranchEnds_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberBranchEnds_MitoSkeleton"]
    )
    df["MitoEnds_Math_Total_NumberNonTrunkBranches_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn"]
    )
    df["MitoEnds_Math_Total_NumberTrunks_MitoSkeleton"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_NumberTrunks_MitoSkeleton"]
    )
    df["MitoEnds_Math_TotalObjectSkeltnLngth_MitoSkeleton_PerCell"] = (
        df["Children_MitoEnds_Count"]
        * df["Mean_MitoEnds_ObjectSkeleton_TotalObjectSkeltnLngth_MtSkltn"]
    )
    df["MitoEnds_Total_ObjectSkeltnLngth_MitoSkeleton_PerCell_Area"] = (
        df["MitoEnds_Math_TotalObjectSkeltnLngth_MitoSkeleton_PerCell"]
        / df["AreaShape_Area"]
    )
    return df

combined_cell_df_mitolyso_merged = calculate_corrected_features(filtered_df.copy())
display(combined_cell_df_mitolyso_merged)
display(
    combined_cell_df_mitolyso_merged[
        [
            "Children_Mitochondria_Count",
            "Mean_Mitochondria_AreaShape_Area",
            "Math_Total_Mitochondria_AreaShape_Area_PerCell",
            "Median_Mitochondria_Area_PerCell_Area",
            "Mean_Mitochondria_Area_PerCell_Area",
            "AreaFraction_Mitochondria_PerCell",
            "Mean_Mitochondria_Distance_Centroid_Minimum_Ratio",
            "Density_Children_Mitochondria_Count_PerCell_Area",
            "Mean_Intensity_Per_Mitochondria_PerCell_Area",
        ]
    ]
)


# Note: use the "Corr_" flag to get the corrected values
# display(combined_cell_df_mitolyso_merged[["Passage Group","Mean_Lysosomes_Intensity_MeanIntensity_LAMP1","MeanIntensity_Mitochondria_PerCell_Ratio","MeanIntensity_Lysosomes_PerCell_Ratio","Mean_Lysosomes_Area_PerCell_Ratio","Mean_Mitochondria_Area_PerCell_Ratio","Math_Total_Mitochondria_AreaShape_Area_PerCell","Math_Total_Lysosomes_AreaShape_Area_PerCell"]])

In [ ]:
# file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'
# pairs_samples = [('P6-8', 'P9-10'), ('P6-8', 'P11-13'), ('P6-8', 'P14-16'), ('P6-8', 'P17-18'),
# ('P6-8', 'P20-21'), ('P6-8', 'P22-24'), ('P9-10', 'P11-13'), ('P9-10', 'P14-16'), ('P9-10', 'P17-18'),
# ('P9-10', 'P20-21'), ('P9-10', 'P22-24'), ('P11-13', 'P14-16'), ('P11-13', 'P17-18'), ('P11-13',
# 'P20-21'), ('P11-13', 'P22-24'), ('P14-16', 'P17-18'), ('P14-16', 'P20-21'), ('P14-16', 'P22-24'),
# ('P17-18', 'P20-21'), ('P17-18', 'P22-24'), ('P20-21', 'P22-24')]
columns_list = define_cell_features(combined_cell_df_mitolyso_merged)


def define_cell_features(df):
    # Get the columns of the dataframe
    columns_list = df.columns.tolist()
    columns_list = [
        col
        for col in columns_list
        if "Metadata" not in col
        and "FileName" not in col
        and "PathName" not in col
        and pd.api.types.is_numeric_dtype(df[col])
    ]
    old_columns_list = columns_list = [
        col
        for col in columns_list
        if "Metadata" not in col and "FileName" not in col and "PathName" not in col
    ]
    print(
        "Original columns:",
        len(old_columns_list),
        "Filtered columns:",
        len(columns_list),
    )
    return columns_list


mito_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Mito" in col or "Mitochondria" in col)
        and ("DAPI" not in col and "LAMP1" not in col and "Frame" not in col)
        and not (col.startswith("Nuclei_"))
    ]
)
lyso_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Lysosome" in col or "LAMP1" in col or "Lyso" in col)
        and ("DAPI" not in col and "Mito" not in col and "Frame" not in col)
        and not(col.startswith("Nuclei_"))
    ]
)
nuc_features = make_feature_dict(
    [
        col
        for col in columns_list
        if ("Nuc" in col or "DAPI" in col)
        and ("MitoTracker" not in col and "LAMP1" not in col and "Frame" not in col)
    ]
)
cell_features = make_feature_dict(
    [
        col
        for col in columns_list
        if "AreaShape" in col
        and "Mito" not in col
        and "Lyso" not in col
        and "LAMP1" not in col
        and "Nuc" not in col
        and "DAPI" not in col
        and "Metadata" not in col
        and "FileName" not in col
        and "PathName" not in col
    ]
)
print(columns_list)


## Feature lists here:

In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features, cell_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features", "Cell Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"- {feature}\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


## Check if there are any mixed types hiding out

In [ ]:
#Debug to check for mixed types in columns
for col in combined_cell_df_mitolyso_merged.columns:
    types = combined_cell_df_mitolyso_merged[col].apply(type).value_counts()
    if len(types) > 1:
        print(f"Column '{col}' has mixed types: {types}")

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"

#print(combined_cell_df_mitolyso_merged.iloc["*","3f9f05979b4d963d3a416940cd146486c35afd73ff8824950945744285caf345b27996985b25de4f280bbe6380ecd3bdb27996985b25de4f280bbe6380ecd3bdaac39e649e9e480b6aa7e043c7ae0740aac39e649e9e480b6aa7e043c7ae0740ebad8781c113930a511fb1e0b33a5ae1ebad8781c113930a511fb1e0b33a5ae1a6ca8186980a3310d2f7e3d35512fcb3a6ca8186980a3310d2f7e3d35512fcb3c3b031fa1953fe9aed603e94b738cd8dc3b031fa1953fe9aed603e94b738cd8df08d58956a263e00e263dbe462b2b8781c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab371c992d8f3dd23b778820570de386ab37b4de0c127359612d367afa1651150c8fb4de0c127359612d367afa1651150c8fb4de0c127359612d367afa1651150c8f6b0e15f67abc6df786bf563bc136c3676b0e15f67abc6df786bf563bc136c3676b0e15f67abc6df786bf563bc136c3674a8539031734babcfd06ff30311dbf064a8539031734babcfd06ff30311dbf065918ba3f0a658eec695b9622d9b20547be25a504ff9c89fe118269e9bacab62e3f51d4573daea38cf97711a9273f5ea5155a754ccadd310a7a08caf6dfb4b2b8ca83e03b639c81a341d51d0c7ddff9ea060bc1e34e6f5eb7e3e6b302803a27ca060bc1e34e6f5eb7e3e6b302803a27ca8f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec2278f3d782190481b6e951c89cc6d8ec22703397b32121e5f601a63e09d3aac334803397b32121e5f601a63e09d3aac33483d056f026c80c5ae4e7432aa1aa370bcadb33dceaa2f3a88b5f8bd16cc89e0c8fe3594b08af4a6536ac08d8b57a23de7e4f270ab1aed64ab67cc2525f78a7b20afffb56f4b746063b8425c7e265e2eaba5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba10a5ae3e58d97a40077c5fdf1b46d9ba105f4f978e715ee311af26d2fdabe1c7135f4f978e715ee311af26d2fdabe1c713e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b8e1865dce5f347eb6f67db5a65479d1b869980026743a730b2867acfd8ad94b1169980026743a730b2867acfd8ad94b11"])
def normalize_to_control(df, feature, norm_column = 'AgeGroup'):
    '''
    Normalize a feature to the control group (AgeGroup = 0) for each plate.
    Args:
        df (DataFrame): The DataFrame containing the feature to be normalized.
        feature (str): The name of the feature column to normalize.
        norm_column (str): The column used to identify the control group (default is 'AgeGroup').'`
    Returns:
        Series: A Series containing the normalized feature values.
    '''
    # Take the t0 df - lowest passage data point
    t0_df = df[df[norm_column]==0]
    treatment_df = df[[feature, norm_column]].copy()

    #calculate the mean
    mean_zero = t0_df[feature].mean()
    # Check for non-numeric values
    if not pd.api.types.is_numeric_dtype(treatment_df[feature]):
        print(f"[normalize_to_control] WARNING: {feature} is not numeric!")
    #now update the column to have all rows dividied by the mean of group 0
    treatment_df["norm_" + feature] = treatment_df[feature] / mean_zero  
    #return the normalized feature columnn
    return treatment_df["norm_" + feature]

def normalize_features(df, feature_list):
    '''
    Normalize the features in the DataFrame to the control (age group 0) for each plate.
    Args:
        df (DataFrame): The DataFrame containing the features to be normalized.
        feature_list (list): A list of feature column names to normalize.
    Returns:
        DataFrame: A DataFrame with normalized features for each plate.
    '''
    # Normalize the features to the control (age group 0) for each plate
    norm_df = df.copy()
    for feature in feature_list:
        #print('Normalizing feature: ', feature, '...', norm_df[feature].values[0])
        norm_df[feature] = normalize_to_control(df, feature)
        #print('Normalized feature: ', feature, '...', norm_df[feature].values[0])
    return norm_df


def apply_feature_normalization(df, feature_dict, curr_plates):
    '''
    Apply feature normalization to the DataFrame for each plate in a list of plates.
    Args:
        df (DataFrame): The DataFrame containing the features to be normalized.
        feature_dict (dict): A dictionary containing lists of feature columns to normalize.
        curr_plates (list): A list of plate names to apply normalization to.
    Returns:
        DataFrame: A DataFrame with normalized features for each plate.
    '''
    # Normalize the features to the control (age group 0) for each plate
    norm_cell_df = df.copy()
    for plate in curr_plates:
        curr_plate_df = norm_cell_df[norm_cell_df['Metadata_Plate'] == plate].copy()
        for feature_type in feature_dict:
            #get the normalized features, locate the corresponding features on the plate, and replace them on that plate to the plate
            curr_plate_features_df = normalize_features(curr_plate_df, feature_dict[feature_type])
            curr_plate_df.loc[:, feature_dict[feature_type]] = curr_plate_features_df[feature_dict[feature_type]].astype(float)
        norm_cell_df.loc[norm_cell_df['Metadata_Plate'] == plate] = curr_plate_df
    return norm_cell_df

norm_cell_df = combined_cell_df_mitolyso_merged.copy()
print("Before conversion: AgeGroup dtype:", norm_cell_df["AgeGroup"].dtype)
print("Unique AgeGroup values:", norm_cell_df["AgeGroup"].unique())

norm_cell_df["AgeGroup"] = pd.to_numeric(norm_cell_df["AgeGroup"], errors='raise')

print("After conversion: AgeGroup dtype:", norm_cell_df["AgeGroup"].dtype)
print("Unique AgeGroup values after conversion:", norm_cell_df["AgeGroup"].unique())
#display(norm_cell_df["AgeGroup"].value_counts())

norm_cell_df_cell = apply_feature_normalization(norm_cell_df, cell_features, curr_plates).copy()
norm_cell_df_nuc = apply_feature_normalization(norm_cell_df_cell, nuc_features, curr_plates).copy()
norm_cell_df_mito = apply_feature_normalization(norm_cell_df_nuc, mito_features, curr_plates).copy()
norm_cell_df_mitolyso = apply_feature_normalization(norm_cell_df_mito, lyso_features, curr_plates)

#watch out - merged df might be clipping off all of the features

In [ ]:
norm_cell_df_mitolyso.loc[norm_cell_df_mitolyso['Passage Group'] == 'P6-10'].describe()
#norm_cell_df_mitolyso.describe()


## Trying out the pivot and groupby functions


In [ ]:
feature_meas = "Mean_Mitochondria_AreaShape_MajorAxisLength"

display(make_single_feature_df(norm_cell_df_mitolyso, 'Passage Group', feature_meas, 'Replicate_Number'))


# It's plotting time


## Most interesting features so far
- Intensity_MassDisplacment_MitoTracker - increase
- Median Mitochondria Location CenterMass Intensity X
  - decrease,splits into bimodal
  - similar for y
  - AreaShape_Center_X and Center_Y also have similar pattern
  - And Location_Center
- Median mitochondria loaction max intensity
  - Same as CenterMass
- Mean Mitochondria Solidity - increase to p29
- Children Mitochondria Count - gradual increase
  - But scales with size - Density is not sig (slight increase)
  - Total mito area increases; complements this
- Mean Mito Centroid distance - increased (more peripheral)
  - But median is much more vairable between batches
- Mean Mito Trunks - decreased
- Median mito area per cell area - increased
### Lysosomes
- Total area also goes up
- rep3 looks like an outlier here
- Also has the location_ maxintensity change and go bimodal
- bimodal-looking intensity
### Nuclei
- Nuc area increases; scales with cell size increases
- solidity, extent down
- Nuc area ratio - slight increase


In [ ]:
def single_feature_super_splitviolinplot(
    data_df,
    x_value="AllGroups",
    y_value="Cell_AreaShape_Area",
    replicate_col_name="Replicate_Number",
    out_dir=Path(""),
    xtitle=None,
    ytitle=None,
    order=None,
    legend=True,
    annotate=False,
    test=None,
    show_hist=False,
    remove_outliers=False,
    ylim=None,
    reps_to_exclude=[],
    shapiro=True,
    show=True
):
    """Make a superplot to do multiple comparisons for a feature between different conditions
    Args:
        data_df_1 (_type_): _description_
        group_avg_df_1 (_type_): _description_
        data_df_2 (_type_): _description_
        group_avg_df_2 (_type_): _description_
        x_value (str, optional): _description_. Defaults to "AllGroups".
        y_value (str, optional): _description_. Defaults to "Cell_AreaShape_Area".
        replicate_col_name (str, optional): _description_. Defaults to "Replicate_Number".
        csv_dir (str, optional): _description_. Defaults to "".
        xtitle (_type_, optional): _description_. Defaults to None.
        ytitle (_type_, optional): _description_. Defaults to None.
    """
    import matplotlib.lines as mlines
    from statannotations.Annotator import Annotator
    from statannotations.stats.StatTest import StatTest
    from pathlib import Path

    if order == None:
        order = get_all_group_order()
    pairs = getpairs(data_df, x_value, order=order)
    print(pairs)

    try:
        bottom_fence = None#np.percentile(data_df[y_value], 0.000001)
        top_fence = None#np.percentile(data_df[y_value], 99.99)
    except ValueError as e:
        print(e)
        top_fence = None
    axlim = (bottom_fence, top_fence)
    if show_hist:
        hist = sns.kdeplot(
            data_df, x=y_value, hue=replicate_col_name, palette="pastel"
        )
        plt.xlim(axlim)
        plt.savefig(f"{Path(out_dir, f"{y_value}_{replicate_col_name}_histogram")}.png")
        plt.show()
        plt.close()

        df_sorted = data_df.sort_values(
            by=[x_value], key=lambda x: x.map(passage_groups_sort_key)
        ).reset_index(drop=True)
        
        import kaleido

        hist2 = px.histogram(
            df_sorted,
            x=y_value,
            color=x_value,
            marginal="box",
            # histnorm='probability density',
            #range_x=(0, top_fence),
        )
        hist2.write_image(
            Path(out_dir, f"{y_value}_histogram.png"), scale=1.5
        )
        hist2.show()

    fig, ax = plt.subplots(figsize=(15, 8))
    # plt.style.use("ggplot")
    sns.set_context("talk", font_scale=1.2)
    sns.set_theme(style="whitegrid")

    feature_df = make_single_feature_df(
        data_df, group=x_value, feature=y_value, replicates=replicate_col_name
    )
    if reps_to_exclude:
        feature_df = feature_df[~feature_df[replicate_col_name].isin(reps_to_exclude)]
        print(f"removing replicates: {reps_to_exclude}")
    
    if remove_outliers is True:
        feature_df = remove_outliers_iqr(feature_df)
        display(feature_df)
    group_avg_df = average_groups_by_plate(
        feature_df, x_value=x_value, y_value=y_value, replicates=replicate_col_name
    )

    ax = super_splitviolinplot_helper(
        feature_df,
        group_avg_df,
        ax,
        x_value,
        y_value,
        title=" ",
        replicate_col_name=replicate_col_name,
        pairs=pairs,
        order=order,
        annotate=annotate,
        test=test,
        shapiro=False
    )

    if legend:
        if shapiro:
            group_avg_df_shapiro = apply_shapiro_wilk_test_to_df(
                group_avg_df,
                feature_meas=y_value,
                replicate_col_name="Replicate_Number",
                alpha=0.05,
            )
            #display(group_avg_df_shapiro)
            ax = annotate_legend_with_shapiro(ax, group_avg_df_shapiro, replicate_col_name)
    else:
        ax.legend_.remove()
    if ytitle is not None:
        ax.set_ylabel(ytitle)
    else:
        ax.set_ylabel(y_value.replace("_"," "))
    if xtitle is not None:
        ax.set_xlabel(xtitle)
    if ylim is None:
        ylim=axlim
    
    ax.set_ylim(ylim)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f"{y_value}_{test}.png"))
    if show:
        plt.show()


order = get_all_group_order()
feature_meas = "MitoEnds_Total_ObjectSkeltnLngth_MitoSkeleton_PerCell_Area"
ylabel = None#"Mitochondria per cell"
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = combined_cell_df_mitolyso_merged.copy()
pallete = "pastel"
remove_outliers = True
reps_to_exclude=[4]
plot_dir = "plots/"

single_feature_super_splitviolinplot(
    combined_cell_df_mitolyso_merged,
    x_value=group,
    y_value=feature_meas,
    replicate_col_name="Replicate_Number",
    xtitle=xlabel,
    ytitle=ylabel,
    out_dir=plot_dir,
    annotate=True,
    order=order,
    test="tukey",
    reps_to_exclude=reps_to_exclude,
    show_hist=True,
    remove_outliers=False
)

In [ ]:
order = get_all_group_order()
ylabel = None
xlabel = "Age Groups"
group = "AllGroups"

pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = norm_cell_df_mitolyso.copy()
pallete = "pastel"
remove_outliers = True
reps_to_exclude=[4]
plot_dir = "plots/"


#iterate and make big ball of plots
big_out_folder = (
    "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/extra_plots/norm"
)
for compartment, feature_dict in zip(feature_names, feature_dicts):
    feature_folder = Path(big_out_folder, compartment)
    Path.mkdir(feature_folder, exist_ok=True)
    for feature_type, features in feature_dict.items():
        newfolder = Path(feature_folder,feature_type)
        Path.mkdir(newfolder, exist_ok=True)
        for feature in features:
            single_feature_super_splitviolinplot(
                this_df,
                x_value=group,
                y_value=feature,
                replicate_col_name="Replicate_Number",
                xtitle=xlabel,
                ytitle=None,
                out_dir=newfolder,
                annotate=True,
                order=order,
                test="dunn",
                reps_to_exclude=reps_to_exclude,
                show_hist=False,
                remove_outliers=False,
                show=False
            )

/home/mattiazzilab/.conda/envs/jupyter/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning:

16.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.



P14-16 vs. P29+: Custom statistical test, P_val:1.500e-03
P14-16 vs. Doxo: Custom statistical test, P_val:5.000e-03
P11-13 vs. P29+: Custom statistical test, P_val:3.100e-03
P11-13 vs. Doxo: Custom statistical test, P_val:1.010e-02
P6-10 vs. P29+: Custom statistical test, P_val:7.000e-04
P6-10 vs. Doxo: Custom statistical test, P_val:2.500e-03
[('P6-10', 'P11-13'), ('P6-10', 'P14-16'), ('P6-10', 'P17-19'), ('P6-10', 'P20-22'), ('P6-10', 'P23-25'), ('P6-10', 'P26-28'), ('P6-10', 'P29+'), ('P6-10', 'Doxo'), ('P11-13', 'P14-16'), ('P11-13', 'P17-19'), ('P11-13', 'P20-22'), ('P11-13', 'P23-25'), ('P11-13', 'P26-28'), ('P11-13', 'P29+'), ('P11-13', 'Doxo'), ('P14-16', 'P17-19'), ('P14-16', 'P20-22'), ('P14-16', 'P23-25'), ('P14-16', 'P26-28'), ('P14-16', 'P29+'), ('P14-16', 'Doxo'), ('P17-19', 'P20-22'), ('P17-19', 'P23-25'), ('P17-19', 'P26-28'), ('P17-19', 'P29+'), ('P17-19', 'Doxo'), ('P20-22', 'P23-25'), ('P20-22', 'P26-28'), ('P20-22', 'P29+'), ('P20-22', 'Doxo'), ('P23-25', 'P26-28'),

/home/mattiazzilab/.conda/envs/jupyter/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning:

16.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.

/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/mitolyso_plot_functions.py:290: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



ANOVA F statistic: 6.007054792151893
ANOVA p value: 0.0001264513744078323
P23-25 vs. P29+: Custom statistical test, P_val:1.500e-03


/home/mattiazzilab/.conda/envs/jupyter/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning:

16.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.



...

P6-10 vs. P23-25: Custom statistical test, P_val:3.790e-02
P6-10 vs. P26-28: Custom statistical test, P_val:2.050e-02
P6-10 vs. P29+: Custom statistical test, P_val:1.690e-02
P6-10 vs. Doxo: Custom statistical test, P_val:1.610e-02


/home/mattiazzilab/.conda/envs/jupyter/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning:

16.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.



[('P6-10', 'P11-13'), ('P6-10', 'P14-16'), ('P6-10', 'P17-19'), ('P6-10', 'P20-22'), ('P6-10', 'P23-25'), ('P6-10', 'P26-28'), ('P6-10', 'P29+'), ('P6-10', 'Doxo'), ('P11-13', 'P14-16'), ('P11-13', 'P17-19'), ('P11-13', 'P20-22'), ('P11-13', 'P23-25'), ('P11-13', 'P26-28'), ('P11-13', 'P29+'), ('P11-13', 'Doxo'), ('P14-16', 'P17-19'), ('P14-16', 'P20-22'), ('P14-16', 'P23-25'), ('P14-16', 'P26-28'), ('P14-16', 'P29+'), ('P14-16', 'Doxo'), ('P17-19', 'P20-22'), ('P17-19', 'P23-25'), ('P17-19', 'P26-28'), ('P17-19', 'P29+'), ('P17-19', 'Doxo'), ('P20-22', 'P23-25'), ('P20-22', 'P26-28'), ('P20-22', 'P29+'), ('P20-22', 'Doxo'), ('P23-25', 'P26-28'), ('P23-25', 'P29+'), ('P23-25', 'Doxo'), ('P26-28', 'P29+'), ('P26-28', 'Doxo'), ('P29+', 'Doxo')]
removing replicates: [4]
[('P6-10', 'P11-13'), ('P6-10', 'P14-16'), ('P6-10', 'P17-19'), ('P6-10', 'P20-22'), ('P6-10', 'P23-25'), ('P6-10', 'P26-28'), ('P6-10', 'P29+'), ('P6-10', 'Doxo'), ('P11-13', 'P14-16'), ('P11-13', 'P17-19'), ('P11-13', 'P

/home/mattiazzilab/.conda/envs/jupyter/lib/python3.13/site-packages/seaborn/categorical.py:3399: UserWarning:

16.7% of the points cannot be placed; you may want to decrease the size of the markers or use stripplot.

/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/mitolyso_plot_functions.py:290: FutureWarning:

The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior



[('P6-10', 'P11-13'), ('P6-10', 'P14-16'), ('P6-10', 'P17-19'), ('P6-10', 'P20-22'), ('P6-10', 'P23-25'), ('P6-10', 'P26-28'), ('P6-10', 'P29+'), ('P6-10', 'Doxo'), ('P11-13', 'P14-16'), ('P11-13', 'P17-19'), ('P11-13', 'P20-22'), ('P11-13', 'P23-25'), ('P11-13', 'P26-28'), ('P11-13', 'P29+'), ('P11-13', 'Doxo'), ('P14-16', 'P17-19'), ('P14-16', 'P20-22'), ('P14-16', 'P23-25'), ('P14-16', 'P26-28'), ('P14-16', 'P29+'), ('P14-16', 'Doxo'), ('P17-19', 'P20-22'), ('P17-19', 'P23-25'), ('P17-19', 'P26-28'), ('P17-19', 'P29+'), ('P17-19', 'Doxo'), ('P20-22', 'P23-25'), ('P20-22', 'P26-28'), ('P20-22', 'P29+'), ('P20-22', 'Doxo'), ('P23-25', 'P26-28'), ('P23-25', 'P29+'), ('P23-25', 'Doxo'), ('P26-28', 'P29+'), ('P26-28', 'Doxo'), ('P29+', 'Doxo')]


In [ ]:
#pairplot for funsies
# sns.pairplot(
#     combined_cell_df_mitolyso_merged,
#     hue="AllGroups",
#     vars=mito_features["radialdistribution"],
#     diag_kind="kde",
#     plot_kws={"alpha": 0.5},
# )
# plt.show()

In [ ]:
#Using one-way anova and Tukey's HSD to compare means of non normalized values
order = get_all_group_order()
feature_meas = "AreaShape_Area"
ylabel = "Area"


group = 'AllGroups'
replicates = 'Replicate_Number'
pairs = getpairs(combined_cell_df_mitolyso, group, order)
this_df = combined_cell_df_mitolyso_merged.copy()
pallete = "pastel"
remove_outliers = True

feature_df = make_single_feature_df(this_df, group=group, feature=feature_meas, replicates='Replicate_Number')
group_avg_df = average_groups_by_plate(feature_df, x_value=group, y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value=group, y_value=feature_meas, replicate_col_name='Replicate_Number')

if remove_outliers is True:
    feature_df = remove_outliers_iqr(feature_df)
    display(feature_df)

display(feature_df)
display(group_avg_df)
display(group_avg_df_pivot)

sns.set_theme(style="ticks")
#sns.set_context("notebook", font_scale=1.9)

plt.figure(figsize=(12, 8))
sns.set_context("talk", font_scale=0.5)
plt.figure(dpi=300)


sns.violinplot(data=feature_df, x=group,
            y=feature_meas,
            order=order,
            fill = False,
            color= 'gainsboro',
            cut=1,
            native_scale=True,
            linecolor='k',
            inner= None,
            #inner_kws=dict(box_width = 5)
            )

ax = sns.swarmplot(data=group_avg_df, x=group,
            y=feature_meas,
            hue = replicates,
            order=order,
            palette=pallete,
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=0.5)

#use a boxplot to draw the mean line - thinking outside the box :)
sns.boxplot(data = group_avg_df, x = group,
            y = feature_meas,
            showmeans=True,
            meanline=True,
            meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
            medianprops={'visible': False},
            whiskerprops={'visible': False},
            zorder=1,
            showfliers=False,
            showbox=False,
            showcaps=False,
            ax = ax)

ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel(group)
if ylabel == None:
    plt.ylabel(feature_meas.replace('_', ' '))


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

# Extract the data for each group

# Perform the one-way ANOVA test

# Print the results

pvalues,pairs = anova_with_tukey_posthoc(
    group_avg_df, x_value=group, y_value=feature_meas, display_results=True
)


annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order)
annotator.configure(text_format='star', loc='inside', verbose = 2, hide_non_significant=True)
annotator.set_pvalues_and_annotate(pvalues)

plt.savefig("plots/new_plots/" + feature_meas + '_anova_superviolinplot.png', dpi=300)
plt.show()



#### For Kruskal-Walis
```python
sns.catplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            fill = False,
            palette='Set2',
            kind = 'violin',
            inner = None)

sns.boxplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            showfliers = False,
            palette='Set2')

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order)

sns.despine()
plt.gcf().set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))
plt.ylim(-0.5, 6)


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

annotator = Annotator(ax, pairs, data=group_avg_df_pivot, x='Passage Group', y=feature_meas, order=order) 0 
annotator.configure(test='Kruskal', text_format='star', loc='inside')
annotator.apply_and_annotate()


plt.savefig(feature_meas + '_boxplot.png', dpi=300)
```


## Make the plots - nonparametric
### ytitles
ytitle = "Distance of Mitochondria from Cell Center"
ytitle = "Distance of Lysosomes from Cell Center"
"Number of Mitochondria per Cell (Relative to Control)"
ytitle = "Number of Lysosomes per Cell (Relative to Control)"
ytitle = "Total Normalized Mitochondrial Area Per Cell"
ytitle = "Normalized # of Mitochondrial Endpoints
ytitle = "Normalized # of Mitochondrial Trunks"
ytitle = "Mean Length of Mitochondrial Network"
ytitle = "Mean # of Mitochondrial Branches"
ytitle = "Minimum Distance of Mitochondria from Cell"
ytitle = "Mean Total Mitochondrial Area"
ytitle = "Mean Mitochondrial Size"
ytitle = "Mean Total Mitochondrial Perimeter"
ytitle = "Mean Mitochondrial Granularity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean LAMP1 Intensity Per Lysosome"
ytitle = "Mean Mitochondria Intensity Per Cell"
ytitle = "Mean Lysosome Intensity Per Cell"
ytitle = "Total LAMP1 Intensity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean MitoTracker Intensity"
ytitle = "Ratio of Mitochondria Area to Cell Area"
ytitle = "Ratio of Lysosome Area to Cell Area"
ytitle = "MitoTracker Intensity Per Cell Area"
ytitle = "LAMP1 Intensity Per Cell Area"
ytitle = "Lysosomal Circularity"
ytitle = "MitoTracker Intensity Per Cell Area"

In [ ]:
#Nonparametric Function version

#Try this too
#norm_cell_df_mitolyso["Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn_PerCell"] = norm_cell_df_mitolyso["Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn"] * norm_cell_df_mitolyso["Children_MitoEnds_Count"]
#feature_meas = "Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn_PerCell"

#feature_meas = "Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn"
feature_meas = "Corr_Area_PerCell_Ratio_Lysosomes" #- RadialDistribution_MeanFrac_MitoTracker_2of4 RadialDistribution_MeanFrac_MitoTracker_3of4 RadialDistribution_MeanFrac_MitoTracker_4of4"
ytitle = "Total Lysosomal Area Per Cell"
xtitle = "Passage Group"

order = order
print(order)

if feature_meas == 'AreaShape_Area':
    norm_cell_df_mitolyso[feature_meas] = normalize_to_control(norm_cell_df_mitolyso,feature_meas)
    ytitle = "Mean Cell Size"



ylimit = None# (-1,2)

pallete = "pastel"
#pallete = "pastel" 
remove_outliers = True

def remove_outliers_iqr(df, cols = None):
    if cols is None:
        cols = df.select_dtypes('number').columns  # limits to a (float), b (int) and e (timedelta)
    df_sub = df.loc[:, cols]

    iqr = df_sub.quantile(0.75, numeric_only=False) - df_sub.quantile(0.25, numeric_only=False)
    
    #calculate  extreme outlisers by dividing median by iqr
    lim = np.abs((df_sub - df_sub.median()) / iqr) < 2.22

    # replace outliers with nan
    df.loc[:, cols] = df_sub.where(lim, np.nan)
    df.dropna(subset=cols, inplace=True) # drop rows with NaN in numerical columns
    return df

def make_superviolinplot_with_kruskal(data, group, feature_meas, replicates, xtitle=None, ytitle = None, pallete='pastel', ylim = None, order = None, remove_outliers = False):
    
    if order is None:
        order = data[group].dropna().unique().tolist()
    
    if ytitle is None:
        ytitle = feature_meas.replace('_', ' ')
    if xtitle is None:
        xtitle = group.replace('_', ' ')

       
    feature_df = make_single_feature_df(data, group=group, feature=feature_meas, replicates=replicates)
     
    if remove_outliers is True:
        feature_df = remove_outliers_iqr(feature_df)
        display(feature_df)
        
    
    pairs = getpairs(feature_df, group, order)


    group_avg_df = average_groups_by_plate(feature_df, x_value=group, y_value=feature_meas, replicates=replicates)
    group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value=group, y_value=feature_meas, replicates=replicates)

    sns.set_theme(style="ticks")
    sns.set_context("talk", font_scale=0.5)

    plt.figure(dpi=300)

    sns.violinplot(data=feature_df, x=group,
                y=feature_meas,
                order=order,
                fill = False,
                color= 'gainsboro',
                cut=2,
                native_scale=True,
                linecolor='k',
                inner= None,
                #inner_kws=dict(box_width = 5)
                )

    ax = sns.swarmplot(data=group_avg_df, x=group,
                y=feature_meas,
                hue = replicates,
                order=order,
                palette=pallete,
                size=10, 
                edgecolor="k", 
                linewidth=1,
                dodge=0.5)
    
    #use a boxplot to draw the mean line - thinking outside the box :)
    sns.boxplot(data = group_avg_df, x = group,
                y = feature_meas,
                showmeans=True,
                meanline=True,
                meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=1,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax = ax)

    ax.legend_.remove()

    sns.despine()
    plt.gcf()#.set_size_inches(10, 6)
    plt.xlabel(xtitle)
    plt.ylabel(ytitle)
    plt.ylim(ylim)

    from statannotations.Annotator import Annotator
    annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order) 
    annotator.configure(test='Kruskal', 
                        text_format='star', 
                        #pvalue_format = 'simple',
                        loc='inside', 
                        hide_non_significant = True,
                        color = 'black',
                        verbose = 2)
    annotator.apply_and_annotate()

    plt.savefig("plots/new_plots/" + feature_meas + '_superviolinplot.png', dpi=300)
    plt.show()
    


make_superplot_with_kruskal(norm_cell_df_mitolyso, 'AllGroups', feature_meas, 'Replicate_Number', xtitle, ytitle, pallete, ylimit, order=order, remove_outliers=True)

### To export the normalized csv:


In [ ]:

norm_cell_df_mitolyso.to_csv(os.path.join('All_Cell_w_metadata_normalized.csv'), index=False)

#  Old anaylysis code - with plotly
```python
boxplot(df, # dataframe name: mito_df, nuclei_df, image_df, outline_df, lysosomes_df
        'Variable', # the variable you wanna plot, column name
        'Y Title',# name of your y-axis (custom)
        'Time Point',#name of your x-axis (custom)
        'Red', # color of the boxes
        save=True, # if wanna save change to False to True, False is default
        save_name='_boxplot.png') #specify the name of the plot that you save

```

## The clustering zone

In [ ]:
from sklearn.decomposition import PCA
filter_key = 'LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA()
components = pca.fit_transform(X)
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

fig = px.scatter_matrix(
    components,
    labels=labels,
    dimensions=range(4),
    color=ordered_cell["Time"]
)
fig.update_traces(diagonal_visible=False)
fig.show()

In [ ]:
from sklearn.decomposition import PCA
filter_key = '_MitoTracker'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA(n_components=3)
components = pca.fit_transform(X)

fig = px.scatter_3d(components, x=0, y=1,z=2, color=ordered_cell['Time'] , labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
})
fig.show()

In [ ]:
from sklearn.manifold import TSNE
filter_key = 'Entropy_LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

tsne = TSNE(n_components=2, random_state=0)
projections = tsne.fit_transform(X)

fig = px.scatter(
    projections, x=0, y=1,
    color=ordered_cell['Time']
)
fig.show()

In [ ]:
ordered_nuc = combined_nuclei_df.sort_values(
  by='Time', 
  ascending=True)

ordered_nuc['Passage Group'] = ordered_nuc['PassageNumber'].apply(passage_group)

feature = 'AreaShape_Solidity'

fig = px.box(ordered_nuc, x=feature, y='Passage Group', color = 'Passage Group', labels={
                     feature: feature.strip('_'),
                     'Passage Group': 'Passage Group'})

fig.show()